# Notebook 01 — Curves and Instruments

**quant-desk-toolkit** · github.com/hyun-quant/quant-desk-toolkit

---

This notebook bootstraps the OIS and SOFR projection curves from market quotes, then prices three instrument types: an Interest Rate Swap, a Bond, and a Repo (SFT).

**Outputs**: `curves.pkl`, `instruments.pkl`


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from curve_factory import Curve, bootstrap_ois_curve, build_sofr_projection_curve
from instruments import InterestRateSwap, Bond, SFT

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
print('Imports OK')

---
## 1. OIS Curve Bootstrap

The OIS curve is bootstrapped sequentially: short-end deposits (simple interest) fix near-term discount factors; longer-maturity OIS swap par rates pin each successive pillar.

The par rate condition: $K^* = (1 - P(0,T_n)) / \sum_i \tau_i P(0,T_i)$, solved for $P(0,T_n)$ given all previous pillars.


In [ ]:
# Market data — representative USD OIS (late-cycle environment ~2023)
deposit_tenors = np.array([1/12, 3/12, 6/12])
deposit_rates  = np.array([0.0480, 0.0500, 0.0510])

ois_swap_tenors = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
ois_swap_rates  = np.array([0.0500, 0.0480, 0.0460, 0.0440, 0.0430, 0.0420])

ois_curve = bootstrap_ois_curve(
    deposit_tenors = deposit_tenors,
    deposit_rates  = deposit_rates,
    swap_tenors    = ois_swap_tenors,
    swap_rates     = ois_swap_rates,
)

print(f'OIS curve: {ois_curve}')
print()
print(f'{"Tenor":>8}  {"DF P(0,T)":>12}  {"Zero Rate":>12}')
print('-' * 36)
for t in [0.5, 1, 2, 3, 5, 7, 10]:
    df = float(ois_curve.df(t))
    zr = float(ois_curve.zero_rate(t)) * 100
    print(f'{t:>7.1f}Y  {df:>12.6f}  {zr:>10.4f}%')

---
## 2. SOFR Projection Curve

Post-LIBOR, the floating leg of a SOFR swap is projected using a *separate* SOFR curve rather than the OIS curve. A small basis (~5-25bp) reflects the credit/liquidity premium in term SOFR over overnight OIS.


In [ ]:
sofr_swap_tenors = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
sofr_swap_rates  = np.array([0.0525, 0.0505, 0.0485, 0.0465, 0.0455, 0.0445])

sofr_curve = build_sofr_projection_curve(
    ois_curve        = ois_curve,
    sofr_swap_tenors = sofr_swap_tenors,
    sofr_swap_rates  = sofr_swap_rates,
)

print(f'SOFR projection curve: {sofr_curve}')
print()
print(f'{"Tenor":>8}  {"OIS fwd 3M":>12}  {"SOFR fwd 3M":>14}  {"Basis (bp)":>12}')
print('-' * 52)
for t in [0.5, 1.0, 2.0, 3.0, 5.0, 7.0]:
    f_ois  = float(ois_curve.forward_rate(t, t + 0.25, compounding='simple')) * 100
    f_sofr = float(sofr_curve.forward_rate(t, t + 0.25, compounding='simple')) * 100
    print(f'{t:>7.1f}Y  {f_ois:>11.4f}%  {f_sofr:>13.4f}%  {(f_sofr-f_ois)*100:>10.2f}')

---
## 3. Curve Visualisation

In [ ]:
t_plot = np.linspace(0.01, 10, 200)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(t_plot, [float(ois_curve.df(t))  for t in t_plot], lw=2, color='steelblue', label='OIS')
axes[0].plot(t_plot, [float(sofr_curve.df(t)) for t in t_plot], lw=2, color='tomato', ls='--', label='SOFR')
axes[0].set_title('Discount Factors P(0,T)'); axes[0].set_xlabel('Tenor (yr)'); axes[0].legend()

axes[1].plot(t_plot, [float(ois_curve.zero_rate(t))*100  for t in t_plot], lw=2, color='steelblue', label='OIS')
axes[1].plot(t_plot, [float(sofr_curve.zero_rate(t))*100 for t in t_plot], lw=2, color='tomato', ls='--', label='SOFR')
axes[1].set_title('Zero Rates (%)'); axes[1].set_xlabel('Tenor (yr)'); axes[1].legend()

axes[2].plot(t_plot[:-1], [float(ois_curve.forward_rate(t, t+0.01))*100  for t in t_plot[:-1]], lw=2, color='steelblue', label='OIS inst. fwd')
axes[2].plot(t_plot[:-1], [float(sofr_curve.forward_rate(t, t+0.01))*100 for t in t_plot[:-1]], lw=2, color='tomato', ls='--', label='SOFR inst. fwd')
axes[2].set_title('Instantaneous Forward Rates (%)'); axes[2].set_xlabel('Tenor (yr)'); axes[2].legend()

plt.tight_layout()
plt.savefig('nb01_curves.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Interest Rate Swap Pricing

The **payer IRS** pays fixed $K$ and receives SOFR floating. Under multi-curve, the floating leg is projected using the SOFR curve but discounted using the OIS curve.

$$V_{payer} = PV_{float} - PV_{fixed}$$

Par rate: $K^* = PV_{float} / (N \cdot \text{Annuity})$


In [ ]:
# 5-year payer IRS, $10mm notional — first find the par rate
_probe = InterestRateSwap(notional=10_000_000, fixed_rate=0.045, tenor=5.0,
                           payment_frequency=2, float_frequency=4, payer=True)
par_rate = _probe.par_rate(ois_curve, sofr_curve)
print(f'5Y SOFR IRS par rate: {par_rate*100:.4f}%')

# Price at par
swap_atm = InterestRateSwap(notional=10_000_000, fixed_rate=par_rate, tenor=5.0,
                             payment_frequency=2, float_frequency=4, payer=True)
pv_atm = swap_atm.pv(ois_curve, sofr_curve)
print(f'At-par payer PV     : ${pv_atm["pv_net"]:,.2f}  (expected ≈ $0)')
print(f'  PV fixed leg      : ${pv_atm["pv_fixed"]:,.2f}')
print(f'  PV float leg      : ${pv_atm["pv_float"]:,.2f}')
print()

# Off-market: fixed 50bp above par — payer is OTM (pays above-market fixed)
swap_otm = InterestRateSwap(notional=10_000_000, fixed_rate=par_rate+0.0050, tenor=5.0,
                             payment_frequency=2, float_frequency=4, payer=True)
pv_otm = swap_otm.pv(ois_curve, sofr_curve)
print(f'Off-market payer (K = par + 50bp):')
print(f'  Net PV            : ${pv_otm["pv_net"]:,.2f}  (negative = liability, paying above-market fixed)')

In [ ]:
# Par rate curve across tenors
tenors_plot = [1, 2, 3, 5, 7, 10]
par_rates_plot = []
for T in tenors_plot:
    s = InterestRateSwap(notional=1e6, fixed_rate=0.05, tenor=float(T),
                         payment_frequency=2, float_frequency=4, payer=True)
    par_rates_plot.append(s.par_rate(ois_curve, sofr_curve) * 100)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tenors_plot, par_rates_plot, 'o-', lw=2, ms=8, color='steelblue')
for x, y in zip(tenors_plot, par_rates_plot):
    ax.annotate(f'{y:.3f}%', (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)
ax.set_title('SOFR IRS Par Swap Rate Curve'); ax.set_xlabel('Tenor (years)'); ax.set_ylabel('Par Rate (%)')
plt.tight_layout(); plt.savefig('nb01_par_rates.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 5. Bond Pricing

Fixed coupon bond: $P_{dirty} = C \sum_i P(0,T_i) + F \cdot P(0,T_n)$

Clean price = dirty price − accrued interest. YTM is the flat rate $y$ such that bond discounted at $y$ equals dirty price.


In [ ]:
bond = Bond(face_value=1_000_000, coupon_rate=0.04, maturity=10.0, coupon_frequency=2)
bond_pv = bond.pv(ois_curve, t_since_last_coupon=0.0)

print('10Y 4% Semi-Annual Bond, $1mm face')
print(f'  Dirty price       : ${bond_pv["dirty_price"]:>12,.2f}')
print(f'  Clean price       : ${bond_pv["clean_price"]:>12,.2f}  (= dirty when AI=0 at coupon date)')
print(f'  YTM (cont. comp.) : {bond_pv["ytm"]*100:.4f}%')
print()

# Mid-period: 3 months after last coupon
bond_pv_mid = bond.pv(ois_curve, t_since_last_coupon=0.25)
print('Same bond, 3 months after last coupon:')
print(f'  Dirty price       : ${bond_pv_mid["dirty_price"]:>12,.2f}')
print(f'  Accrued interest  : ${bond_pv_mid["accrued_interest"]:>12,.2f}  (= 1mm x 4% x 0.25 x 0.5 period)')
print(f'  Clean price       : ${bond_pv_mid["clean_price"]:>12,.2f}')

In [ ]:
# Price-yield curve — positive convexity
yields = np.linspace(0.01, 0.09, 100)
prices = []
for y in yields:
    t_nodes = np.array([0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
    dfs     = np.exp(-y * t_nodes)
    flat    = Curve(tenors=t_nodes, discount_factors=dfs)
    b       = Bond(face_value=100, coupon_rate=0.04, maturity=10.0, coupon_frequency=2)
    prices.append(b.dirty_price(flat))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(yields*100, prices, lw=2.5, color='steelblue')
ax.axvline(x=4.0, color='tomato', ls='--', lw=1.5, label='Coupon = 4% → par at YTM=4%')
ax.axhline(y=100, color='gray', ls=':', lw=1)
ax.set_title('Bond Price vs YTM  (10Y 4% Semi-Annual)')
ax.set_xlabel('YTM (%)'); ax.set_ylabel('Price (% of face)'); ax.legend()
plt.tight_layout(); plt.savefig('nb01_bond_priceyield.png', dpi=120, bbox_inches='tight'); plt.show()
print('Positive convexity: the price-yield curve is bowed outward (convex).')

---
## 6. Repo / SFT Pricing

A **repo** is a collateralised borrowing. The seller posts bonds and agrees to repurchase at a fixed price (= notional × exp(r × T)). The **haircut** ensures the collateral value exceeds the cash by a buffer. A **margin call** is triggered when that buffer erodes.


In [ ]:
sft = SFT(
    notional         = 10_000_000,
    repo_rate        = 0.0530,
    tenor            = 30 / 365,
    haircut          = 0.02,
    is_repo          = True,
    collateral_value = 10_204_082,
)

repurchase_price = sft.repurchase_price()
print(f'Repo: $10mm, 30-day, 5.30% rate, 2% haircut')
print(f'Repurchase price      : ${repurchase_price:,.2f}')
print(f'Repo interest         : ${repurchase_price - sft.notional:,.2f}')
print()

collateral_today = 10_204_082 * (1 - 0.03)
mtm_exp = sft.mtm_exposure(collateral_today, t=15/365)
print(f'After 15 days, collateral -3%:')
print(f'  Collateral MV       : ${collateral_today:,.2f}')
print(f'  MTM exposure        : ${mtm_exp:,.2f}')
print()

margin_call = sft.margin_call(
    collateral_today, t=15/365,
    threshold=50_000,
    minimum_transfer_amount=10_000
)
print(f'  Margin call amount  : ${margin_call:,.2f}')

---
## 7. Save Outputs

In [ ]:
curves_data = {
    'ois_curve'        : ois_curve,
    'sofr_curve'       : sofr_curve,
    'deposit_tenors'   : deposit_tenors,
    'deposit_rates'    : deposit_rates,
    'ois_swap_tenors'  : ois_swap_tenors,
    'ois_swap_rates'   : ois_swap_rates,
    'sofr_swap_tenors' : sofr_swap_tenors,
    'sofr_swap_rates'  : sofr_swap_rates,
}
with open('curves.pkl', 'wb') as f:
    pickle.dump(curves_data, f)
print('Saved curves.pkl')

instruments_data = {
    'swap_notional'      : swap_atm.notional,
    'swap_fixed_rate'    : swap_atm.fixed_rate,
    'swap_tenor'         : swap_atm.tenor,
    'swap_pay_freq'      : swap_atm.payment_frequency,
    'swap_float_freq'    : swap_atm.float_frequency,
    'swap_payer'         : swap_atm.payer,
    'swap_par_rate'      : par_rate,
    'swap_pv_net'        : pv_atm['pv_net'],
    'swap_payment_dates' : swap_atm._fixed_payment_dates(),
    'bond_face'          : bond.face_value,
    'bond_coupon'        : bond.coupon_rate,
    'bond_maturity'      : bond.maturity,
    'bond_freq'          : bond.coupon_frequency,
    'bond_dirty_price'   : bond_pv['dirty_price'],
    'bond_ytm'           : bond_pv['ytm'],
    'sft_notional'       : sft.notional,
    'sft_repo_rate'      : sft.repo_rate,
    'sft_tenor'          : sft.tenor,
    'sft_haircut'        : sft.haircut,
}
with open('instruments.pkl', 'wb') as f:
    pickle.dump(instruments_data, f)
print('Saved instruments.pkl')
print('\nProceed to: notebook_02_monte_carlo.ipynb')